In [1]:
"""
Module - Menu.ipynb
Programmer: F329597
Description - Video Games rental application GUI
Usage - GUI file to be used with Jupyter notebook
"""

import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath('scripts'))

import ipywidgets as widgets
import matplotlib.pyplot as plt
from ipywidgets import GridspecLayout
from scripts.gameSearch import search_games
from gameRent import renting_game
from gameReturn import return_game
from gameSelect import (calculate_weights_for_games,
                        calculate_combined_score,
                        get_values_for_bar_chart,
                        rental_history_filtered_by_date,
                        coordinate_process,
                        values_for_pie_and_bar_chart,
                        bar_chart_count_per_title
                        )

# Output area for displaying results and options
output_area = widgets.Output()

def create_header_cell(text):
    """Generate a styled header cell
    
       Parameters: 
           cell header text
    """
    # header_color="Indigo"
    header_color = '#054fb9'
    return widgets.HTML(value=f"<div style='background-color:{header_color}; \
                        color:white; \
                        padding:5px; \
                        line-height:1.5;'> \
                        <b>{text}</b></div>")


def create_row_cell(cell_data, cell_color):
    """ Generate a styled row cell
    
        Parameters: 
            cell data
            cell color                      
    """
    return widgets.HTML(value=f"<div style='background-color:{cell_color}; \
                        padding:2px; \
                        margin:0; \
                        line-height:1.5;'> \
                        {cell_data}</div>")


def create_availability_cell(value, cell_color, cell_text_color):
    """Generate a styled availability cell with color coding
    
       Parameters: 
            cell data
            cell color
            cell text color                   
    """
    return widgets.HTML(
        value=f"<div style='background-color:{cell_color}; \
        padding:2px;margin:0;line-height:1.5;color:{cell_text_color};'> \
        <b>{value}</b></div>"
    )


def safe_int(value):
    """Safely convert a value to an integer, ensuring that the 
       conversion does not result in errors or unexpected behavior 
       
    Parameters:
        value (int)
        
    Returns:
        value (int)
        0 (int)
    """
    try:
        return int(value)
    except (ValueError, TypeError):
        return 0
        

def on_rent_click(b, game_id, customer_id):
    """Event handler for the Rent Game button
    
    Parameters:
        Widget id (button)
        Game Id (str)
        Customer Id (str)
    """
    with output_area:
        output_area.clear_output()  # Clear previous output

        # execute rent game function
        result = renting_game(customer_id.strip(), game_id.strip())
        display(widgets.HTML(f"<b style='color:red;'>{result}</b>"))


def on_return_click(b, game_id):
    """Event handler for the Rent Game button
    
    Parameters: 
        Widget id (button)
        Game Id (str)
    """
    with output_area:
        output_area.clear_output()  # Clear previous output

        # execute return game function
        result = return_game(game_id.strip())
        display(widgets.HTML(f"<b style='color:red;'>{result}</b>"))


def on_recommend_title_click(b, budget, period, price_weight, popularity_weight):
    """Event handler for the Recommend title button
    
    Parameters: 
        budget value (int)
        period (str)
        price weight (int)
        popularity weight (int)
    """
    with output_area:
        output_area.clear_output()  # Clear previous output
        
        error_message, recommended_games,_ = coordinate_process(period, 
                                             popularity_weight, 
                                             price_weight, 
                                             safe_int(budget))

        # Display results in a grid if matches are found
        if recommended_games:
            num_columns = 3
            num_rows = len(recommended_games) + 1  # Adding 1 for header row

            grid_title = widgets.HTML("<h3>Game recommendation</h3>")
            
            # Create a GridspecLayout to display the results
            grid = GridspecLayout(num_rows, num_columns, width='100%')

            cell_titles = ['Title', 'Copies to Purchase'] 
            # Assign header cells label to the grid
            for i, title in enumerate(cell_titles, start = 0):
                grid[0, i] = create_header_cell(title)
                        
            # Populate the grid with recommeded games data 
            # with alternating row colors
            for i, game in enumerate(recommended_games, start=1):

                # Apply alternating background colors to the game rows                
                row_color = 'WhiteSmoke' if i % 2 == 0  else 'AliceBlue'
                
                # set cell values
                for x, (game, copies) in enumerate(recommended_games.items(), 
                                                   start=1):        
                    # Set cell values
                    grid[x, 0] = create_row_cell(game, row_color)
                    grid[x, 1] = create_row_cell(copies, row_color)

            filtered_history_lines = rental_history_filtered_by_date(period)
            pop, prices = calculate_weights_for_games(filtered_history_lines)
            combined_score = calculate_combined_score(popularity_weight,
                                                  price_weight, 
                                                  pop, 
                                                  prices)

            # Convert the matplotlib figure to a widget
            output = widgets.Output()
            with output:

                chart1 = plot_weighted_scores(period, price_weight, 
                                              popularity_weight,combined_score)
                plt.show(chart1)
                
            # Arrange the grid and bar charts side by side
            hbox1 = widgets.HBox([grid])
            hbox2 = widgets.HBox([output])
            
            # Display the combined layout
            display(grid_title, hbox1, hbox2)
        else:
            display(widgets.HTML(f"<b style='color:red;'>{error_message}</b>"))

def on_title_analysis_click(b):
    """Event handler for the title analysis button
    
    Parameters: 
      Widget id (button)
    """
    with output_area:
        output_area.clear_output()  # Clear previous output
        
        #Convert the matplotlib figure to a widget
        output = widgets.Output()
        with output:              
            chart = create_bar_chart_count_per_title()
            plt.show(chart)
            
        display(output)

def on_genre_analysis_click(b):
    """Event handler for the genre analysis button
    
    Parameters: 
         Widget id (button)
    """
    with output_area:
        output_area.clear_output()  # Clear previous output
        genres, rents, \
        max_genre, \
        title_rents, \
        game_genre_dict, game_title_dict = values_for_pie_and_bar_chart()
        
        # Create the first plot
        output1 = widgets.Output()
        with output1:
            fig1 = create_pie_chart(rents, genres)
            plt.show(fig1)
    
        # Create the second plot
        output2 = widgets.Output()
        with output2:
            fig2 = create_bar_chart(game_title_dict, title_rents, game_genre_dict, max_genre)
            plt.show(fig2)
            
        # Display the plots side by side
        display(widgets.HBox([output1, output2]))
        #display(output)


def on_search_click(b, game_name):
    """Event listener for search button click
       lists all games matching search criteria
       or displays an error message
    
    Parameters:
        Widget id (button)
        Game Name (str)
    """

    with output_area:
        output_area.clear_output()  # Clear previous output

        # Get search query to filter the game database
        search_value = game_name.strip().lower()

        # execute search game function
        matching_games, error_text = search_games(search_value)

        # Display results in a grid if matches are found
        if matching_games:
            num_columns = 7 
            num_rows = len(matching_games) + 1  # Adding 1 for header row

            # Create a GridspecLayout to display the results
            grid = GridspecLayout(num_rows, num_columns, width='100%')            
            
            cell_titles = ['ID', 'Platform', 'Genre', 
                           'Title', 'Price £', 'Purchase Date', 'Availability']         
            # Assign header cells label to the grid
            for i, title in enumerate(cell_titles, start = 0):
                grid[0, i] = create_header_cell(title)
                        
            # Populate the grid with matching games data 
            # with alternating row colors
            game_csv_header = ['ID', 'Platform', 'Genre',
                               'Title', 'Purchase Price £', 'Purchase Date']
            for i, game in enumerate(matching_games, start=1):
                
                # Apply alternating background colors to the game rows                
                row_color = 'WhiteSmoke' if i % 2 == 0  else 'AliceBlue'
                
                # set cell values
                for x, item in enumerate(game_csv_header, start = 0):
                    grid[i, x] = create_row_cell(game[item], row_color)

                # Color code based on game availability                  
                if game['Availability'] == "Available" :
                     cell_text_color = 'Green' 
                else :
                    cell_text_color = 'Red' 
                grid[i, 6] = create_availability_cell(game['Availability'], 
                                                      row_color, 
                                                      cell_text_color)    
            # display built grid
            display(grid)
            
        else:
            display(widgets.HTML(f"<b style='color:red;'>{error_text}</b>"))
            

def update_sliders(change, slider_to_update):
    """Update the value of a slider based on a change event.

    Parameters:
        change (dict) - A dictionary containing the change event data. 
                   It should have a key 'new' representing the new value.
        slider_to_update (Slider) - The slider object whose value needs 
                   to be updated.
    """
    slider_to_update.value = 1 - change['new']
    

def create_pie_chart(rents, genres):
    """Generate pie chart for given data
    
    Parameters:
        rents (dict)
        genres (list)
        
    Returns:
        Figure (graph)
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    wedges, _, autotexts = ax.pie(rents,
                                      autopct="%1.1f%%", 
                                      startangle=90,
                                      colors=plt.cm.tab10.colors,
                                      wedgeprops=dict(edgecolor="black", linewidth=1.5)
                                     )
    ax.legend(wedges, genres, title="Genres", loc="center left", bbox_to_anchor=(1, 0, 0.2, 1))
    ax.set_title("Popularity per Genre")

    return fig

def create_bar_chart(game_title_dict, title_rents, game_genre_dict, max_genre):
    """Generate bar chart for given data
    
    Parameters:
        game_title_dict = {ID: Title}
        title_rents = A dictionary {title: count} (dict)
        game_genre_dict = {ID: Genre} (dict)
        max_genre = genre with the highest number of rents (str)
        
    Returns:
        Figure (graph)
    """
    
    fig, ax = plt.subplots(figsize=(6, 6))
    max_genre_titles = {}
    title_to_genre_id = {}

    for genre_id, title in game_title_dict.items():
        title_to_genre_id[title] = genre_id

    for title, count in title_rents.items():
        genre_id = title_to_genre_id.get(title)
        if genre_id is not None and game_genre_dict.get(genre_id) == max_genre:
            max_genre_titles[title] = count

    titles = list(max_genre_titles.keys())
    counts = list(max_genre_titles.values())

    ax.barh(titles, counts, color='C0', alpha=0.7)
    ax.set_title(f"Breakdown of \"{max_genre}\" by Title")
    ax.set_xlabel("Rental Count")

    return fig
    

def create_bar_chart_count_per_title():
    """Creates a bar chart with title on the x-axis and rental
       count on the y-axis.
       
    Returns:
        Figure (graph)
    """
    rentals = get_values_for_bar_chart()
 
    y = []
    for key in rentals:
        y.append(rentals[key])
    x = list(rentals.keys())

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.bar(x, y, label="Number of rents by title", color="orange")
    ax.set_xlabel("Title")
    ax.set_ylabel("Rentals")
    ax.set_xticks(range(len(x)))  # Set the positions of the ticks
    ax.set_xticklabels(x, rotation=90, ha="right")  # Set the labels and rotate them
    fig.tight_layout()
    
    return fig


def plot_weighted_scores(months, price_weight, popularity_weight, 
                         combined_score):
    """Plots a bar chart with title on the x-axis and combined
       score on the y-axis
       
    Parameters:
        price_weight - The weighting decided by the user (How
                         important price is to them) (int)
        popularity_weight - The weighting decided by the user (How
                            important popularity is to them) (int)
    Returns:
        Figure (graph)
    """
    
    y = []
    for key in combined_score:
        y.append(combined_score[key])
    x = list(combined_score.keys())   
    
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.bar(x, y, label="Weighted score", color="green")
    ax.set_xlabel("Title")
    ax.set_ylabel("Score")
    ax.set_xticks(range(len(x)))  # Set the positions of the ticks
    ax.set_xticklabels(x, rotation=90, ha="right")  # Set the labels and rotate them
    ax.set_title("Best Game Choices: Balancing Popularity \n"
             "and Price Based on Your Preferences")
    ax.legend(["Combined Score"], loc="upper right")
    fig.tight_layout()

    return fig


def style_button(button):
    """Apply style to a button   
    """
    button.style = {
        'button_color': '#4681f4',
        'font_weight':  'bold',
        'text_color' : 'white'
    }


def create_float_slider():
    """Create a fload slider and apply attributes
    
    Returns:
        Floatslider
    """
    slider = widgets.FloatSlider(
        value=0.5,
        min=0.0,
        max=1.0,
        step=0.01,
        continuous_update=True,
        layout=widgets.Layout(width='200px')
    )
    return slider
    

def setup_tab(index):
    """Set up menu option as tabs and it's childrens
    
    Parameters:
        index (int): The index of the tab to set up.
    """
    
    if index == 0:  # search game
        search_text = widgets.Text(description="Search Game:")
        search_button = widgets.Button(description="Search")
        style_button(search_button)
        
        # Attach an event handler to the return button
        search_button.on_click(lambda b: on_search_click(b, search_text.value))

        # Create VBox containers
        vbox1 = widgets.VBox([search_text, search_button])
        
        # Place the vbox container in HBox widgets 
        hbox = widgets.HBox([vbox1],
                           layout=widgets.Layout(border='inset 2px',
                                                 padding='10px'))

        children[0].children = [hbox, output_area]

    elif index == 1: #rent game
        game_id = widgets.Text(description="Game Id:")
        customer_id = widgets.Text(description="Customer Id:")
        rent_button = widgets.Button(description="Rent")
        style_button(rent_button)
        
        # Attach an event handler to the rent button
        rent_button.on_click(lambda b: on_rent_click(b, game_id.value, 
                                                     customer_id.value))

        # Create VBox containers
        vbox1 = widgets.VBox([game_id, customer_id, rent_button])
        
        # Place the vbox container in HBox widgets 
        hbox = widgets.HBox([vbox1],
                           layout=widgets.Layout(border='inset 2px',
                                                 padding='10px'))
        
         # Assign to children to the container
        children[1].children = [hbox, output_area]

    elif index == 2: #return game
        game_id = widgets.Text(description="Game Id:")
        return_button = widgets.Button(description="Return")
        style_button(return_button)

        # Attach an event handler to the return button
        return_button.on_click(lambda b: on_return_click(b, game_id.value))
        
        # Create VBox containers
        vbox1 = widgets.VBox([game_id, return_button])
        
        # Place the vbox container in HBox widgets 
        hbox = widgets.HBox([vbox1],
                           layout=widgets.Layout(border='inset 2px',
                                                 padding='10px'))
                            
         # Assign to children to the container
        children[2].children = [hbox, output_area]

    elif index == 3: # Title recommendation & charts and graphs
        budget = widgets.Text(layout=widgets.Layout(width='100px'))                             
        period = widgets.Dropdown(
                options={' ' : 'All' ,
                         'Last 6 Months' : '6', 
                         'Last 12 Months' : '12'},
                layout=widgets.Layout(width='130px')
                )
         
         # Create two FloatSliders
        slider1 = create_float_slider()
        slider2 = create_float_slider()
        
       # Link the sliders to the external function
        slider1.observe(lambda change: update_sliders(change, slider2), 
                        names='value')
        slider2.observe(lambda change: update_sliders(change, slider1), 
                        names='value')

        # Create labels for alignment
        label1 = widgets.Label(value='Budget:', 
                               layout=widgets.Layout(width='130px'))
        label2 = widgets.Label(value='Period:', 
                               layout=widgets.Layout(width='130px'))
        label3 = widgets.Label(value='Price Weighting:', 
                               layout=widgets.Layout(width='130px'))
        label4 = widgets.Label(value='Poupularity Weighting:', 
                               layout=widgets.Layout(width='130px'))
        
        # Align sliders using HBox and VBox
        aligned_Textbox = widgets.HBox([label1, budget], 
                                layout=widgets.Layout(align_items='center'))
        aligned_Dropdown = widgets.HBox([label2, period], 
                                layout=widgets.Layout(align_items='center'))       
        aligned_slider1 = widgets.HBox([label3, slider1], 
                                layout=widgets.Layout(align_items='center'))
        aligned_slider2 = widgets.HBox([label4, slider2], 
                                layout=widgets.Layout(align_items='center'))
        
        #create buttons
        recommend_game_button = widgets.Button(description="Recommend Game")
        analysis_by_title_button = widgets.Button(description="By Title")
        analysis_by_genre_button = widgets.Button(description="By Genre")
                
        #apply style to the button
        style_button(recommend_game_button)
       
        #ensure button displays full text and make all button same size 
        #on this menu/tab
        style_button(analysis_by_title_button)
        analysis_by_title_button.layout.width = 'auto'
        style_button(analysis_by_genre_button)
        analysis_by_genre_button.layout.width = 'auto'

        # Attach an event handler to the title and Genre buttons
        recommend_game_button.on_click(lambda b: 
                                on_recommend_title_click(b,
                                                   budget.value, 
                                                   period.value, 
                                                   slider1.value, 
                                                   slider2.value))
        analysis_by_title_button.on_click(lambda b: 
                                on_title_analysis_click(b))
                                                   
        analysis_by_genre_button.on_click(lambda b: 
                                on_genre_analysis_click(b))
                                             
        
        # Create VBox containers
        vbox1 = widgets.VBox([aligned_Textbox, 
                              aligned_Dropdown, 
                              aligned_slider1, 
                              aligned_slider2,
                             recommend_game_button],                             
                             layout=widgets.Layout(margin='10px 0 10px 0'))
     
        vbox2_header = widgets.HTML(
                        value="<h3 style='margin:0;'>Graphical analysis</h3>")
        vbox2 = widgets.VBox([vbox2_header,
                            analysis_by_title_button,
                            analysis_by_genre_button],
                            layout=widgets.Layout(margin='10px 0 10px 0', 
                                            border='solid 2px black', 
                                            padding='10px'))
        
        # Place them side by side using HBox
        hbox = widgets.HBox([vbox1, vbox2],
                           layout=widgets.Layout(border='inset 2px', 
                                                 padding='10px'))
        
        # Assign to children to the container
        children[3].children = [hbox, output_area]

        
# Create a Tab widget for the menu
tab = widgets.Tab()

tab_titles = ['Search Game', 'Rent Game', 'Return Game', 'Recommendation']
children = [widgets.VBox(), widgets.VBox(), widgets.VBox(), widgets.VBox()]

tab.children = children

# set each tab title
for i, title in enumerate(tab_titles):
    tab.set_title(i, title)
    tab.style = {'font_weight': 'bold', 
                 'font_size': '16px',
                 'background_color': 'lightblue'
    }


def on_tab_change(change):
    """To handle tab change event, check if the change event is 
       releated to selected tab index
       
    Parameter:
        event that triggered the function 
    """
    if change['name'] == 'selected_index':
        for i, child in enumerate(children):
            if i != change['new']:  # Clear non-selected tabs
                child.children = []
        setup_tab(change['new']) # set up items for selected tab


# Observe tab changes
tab.observe(on_tab_change, names='selected_index')
   
header = widgets.HTML(value=f"<h1 style='text-align:center; \
                            color:green;'>Video Game Rental system</h1>")

# set up first tab items
setup_tab(0)

# Display the application with the tabs and output area
app = widgets.VBox([header, tab])
app